# SafeOps GraphRAG: 검색·답변·RAGAS 검증

기본 문서 질문 4개, 위험 상황 질문 15개, Outside 1개로 총 20문항을 검증합니다.

- 운영 검색은 FAISS와 출처 기반 관계 그래프를 결합합니다. 물질 → 신체 주의사항 → 대응·보고 → 담당 부서·병원 근거를 함께 선택합니다.
- 위험기호는 위험성·주의사항의 내부 연결 근거로 사용하며 답변에는 기호 자체를 출력하지 않습니다.
- 관련성 평가는 한국어 역질문을 기본으로 사용합니다. 기존 영어 역질문 결과와 한국어 진단은 별도 정책·파일로 보존합니다.
- 아래 00~07절은 재현 가능한 평가입니다. 08절 이후는 선택적 과거 실험 부록입니다.
- 기본값은 외부 API 호출 꺼짐입니다. 저장된 결과가 없으면 미실행으로 표시합니다.
- 상세 설계와 이전 측정 결과: [GraphRAG 안내](../docs/GRAPH_RAG.md).

Git에는 코드·설정·검증 질문·이 원본 노트북을 보관합니다. 개인별 생성 답변·평가 출력·캐시·완성 보고서는 `pipeline_artifacts/` 및 로컬 결과 파일로 관리합니다.


## 생성 입력과 평가 자료의 경계

생성 입력은 **질문 + 현재 인덱스에서 검색한 원문·그래프 근거와 연결 경로**입니다.
모범답안은 평가자에게만 전달합니다. 실제 답변이 없으면 미실행으로 표시합니다.

실행 기록은 인덱스 버전·프롬프트 버전·그래프 설정·생성 모델로 구분합니다.


## 00. 커널 준비

이 프로젝트에 `.venv`를 준비했습니다. VS Code 오른쪽 위 **커널 선택 → Python 환경 → industrial-safety-ai/.venv/bin/python**을 선택하세요.

다른 컴퓨터에서는 프로젝트 루트에서 다음과 같이 준비할 수 있습니다.

```bash
python3.11 -m venv .venv
.venv/bin/python -m pip install -r notebooks/requirements.txt
```

노트북은 기존 FAISS를 읽습니다. 문서를 다시 임베딩하지 않습니다. 노트북의 04단계부터 실제 API 비용이 발생할 수 있으므로 `RUN_LIVE`와 캐시 설정을 확인하세요.


In [ ]:
from pathlib import Path
import sys, os, json, csv, hashlib
from IPython.display import display, HTML, Markdown

# 프로젝트 루트 또는 notebooks 폴더에서 열 수 있습니다.
ROOT_OVERRIDE = None  # 자동 탐색 실패 시 프로젝트 절대경로를 입력하세요.
if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / 'pipeline/config.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('industrial-safety-ai 프로젝트를 열거나 ROOT_OVERRIDE를 지정하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 실행 중인 커널의 의존성을 먼저 검사합니다. API 키 값은 출력하지 않습니다.
from importlib.util import find_spec
print('현재 Python:', sys.executable)
missing = [name for name in ('dotenv', 'httpx', 'pypdf', 'langchain_openai', 'faiss')
           if find_spec(name) is None]
if missing:
    raise RuntimeError(
        '현재 노트북 커널에 필요한 패키지가 없습니다: ' + ', '.join(missing)
        + '\nVS Code 오른쪽 위 커널 선택에서 ' + str(ROOT / '.venv/bin/python')
        + '을 선택하고 커널을 재시작한 뒤 모두 실행하세요.'
    )

# 이 노트북 실행은 기본적으로 LangSmith 자동 추적/업로드를 하지 않습니다.
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ.pop('LANGCHAIN_TRACING_V2', None)  # 이전 커널의 구형 설정 제거
from pipeline.pdf_pipeline import config_at, read_json, resolve_pointer, digest
from pipeline import notebook_evaluation as ev

from dotenv import load_dotenv
from pipeline.knowledge_graph import graph_from_folder
CFG = config_at(ROOT / 'pipeline/config.json')
ENV_FILE = Path(os.getenv('SAFEOPS_ENV_FILE', str(ROOT / '.env'))).expanduser().resolve()
load_dotenv(ENV_FILE, override=False)
INDEX_READY = (CFG['output_root'] / 'current.json').exists()
INDEX_DIR = resolve_pointer(CFG['output_root'], 'current.json') if INDEX_READY else None
INDEX_MANIFEST = read_json(INDEX_DIR / 'manifest.json') if INDEX_READY else None
# 원본 PDF에서 로컬 준비만 수행하며 임베딩/LLM을 호출하지 않습니다.
if not (CFG['output_root'] / 'prepared.json').exists():
    from pipeline.pdf_pipeline import prepare
    prepare(CFG)
PREPARED_DIR = resolve_pointer(CFG['output_root'], 'prepared.json')
GRAPH = graph_from_folder(INDEX_DIR or PREPARED_DIR)
print('프로젝트:', ROOT.name)
print('OpenAI 키 설정:', bool(os.getenv('OPENAI_API_KEY')))
print('그래프:', GRAPH['statistics'])
print('인덱스:', INDEX_MANIFEST['build_id'] if INDEX_READY else '미구축 — prepare 완료, build 필요')
if INDEX_READY:
    print('문서 임베딩:', INDEX_MANIFEST['embedding_model'])
    print('청크 수:', INDEX_MANIFEST['chunk_count'], '| 벡터 차원:', INDEX_MANIFEST['vector_dimension'])


## 01. 질문과 실행 설정

`evaluation/notebook_cases.json`의 현재 질문을 읽습니다. 문항 수와 유형은 아래에서 자동 집계합니다.
Outside는 Q05a 한 문항입니다. 위험 상황 질문은 문서 안의 공통 대응과 추가 정보 확인을 함께 평가합니다.

- TOP_K: 벡터 검색 결과 수. 그래프 근거는 별도로 추가됩니다.
- RUN_LIVE=True: 질문 임베딩과 답변 생성. USE_CACHE=True이면 같은 조건의 완료 캐시를 재사용합니다.
- 같은 질문으로 다시 생성하려면 USE_CACHE=False를 사용합니다.
- 모범답안·위험 태그·검증 초점·허용 상태는 생성 모델에 전달하지 않습니다.


In [ ]:
CASES_PATH = ROOT / 'evaluation/notebook_cases.json'
CASESET = read_json(CASES_PATH)
CASES = CASESET['cases']
QUESTIONS_ONLY = [{k: c[k] for k in ('id', 'question', 'difficulty')} for c in CASES]
REVIEW_VERSION = ev.review_version(CASES)
TOP_K = 5
ANSWER_MODEL = 'gpt-4o'  # 교수 제공 코드와 동일한 생성 모델을 기준선으로 사용
RUN_LIVE = os.getenv('SAFEOPS_RUN_LIVE', '0') == '1'  # 직접 실행 시 True로 변경
USE_CACHE = True
assert CASES and len({c['id'] for c in CASES}) == len(CASES)
assert sum(c['difficulty'] == 'Outside' for c in CASES) == 1
from collections import Counter
print('데이터셋:', CASESET.get('dataset_version'), '| 문항 수:', len(CASES))
print('구성:', dict(Counter(c['difficulty'] for c in CASES)))
for c in CASES:
    print(f"{c['id']} [{c['difficulty']} / {c.get('category', '')}] {c['question']}")


## 02. 모범답안 초안 검토

질문별 reference_answer, required_points, forbidden_claims, evidence를 확인하세요.
**아래 초안을 사람이 작성한 정답으로 표시하지 않습니다.** 사람은 PDF 원본과 대조한 후
`evaluation/notebook_cases.json`에서 review_status를 approved로 바꾸고 reviewer와 reviewed_at을 입력합니다.
내용을 고쳤더라도 reference_author의 AI_draft 작성 이력을 유지하고 검토자 기록을 추가하세요.


In [ ]:
for c in CASES:
    display(Markdown(f"### {c['id']} — {c['difficulty']}"))
    print('질문:', c['question'])
    print('검토용 모범답안:', c['reference_answer'])
    print('필수 내용:', ' / '.join(c['required_points']))
    print('금지 주장:', ' / '.join(c['forbidden_claims']))
    print('기대 상태(초안):', c['expected_status'], '| 검토 상태:', c['review_status'])


## 03. 정답 근거 후보의 페이지 원문 확인

아래는 검색 결과가 아닌, 정답 초안 작성에 사용한 페이지 후보입니다.
오른쪽 실제 답변과 비교하기 전에 이 원문과 PDF 자체를 확인해 정답 기준을 확정하세요.
추출문에는 표나 이미지 내용이 빠질 수 있습니다. PDF 번호는 1부터 시작합니다.
Outside 질문은 정답 페이지가 없고, 적절한 보류를 평가합니다.


In [ ]:
# 한 번에 확인할 질문 ID를 바꿔가며 원문을 검토하세요.
REVIEW_QUESTION_ID = 'Q01'
case = next(c for c in CASES if c['id'] == REVIEW_QUESTION_ID)
for page in ev.reference_preview(CFG, case):
    display(Markdown(f"**{page['filename']} / PDF {page['pdf_page']}쪽**"))
    source_path = CFG['input_root'] / page['filename']
    display(Markdown(f"[PDF 원본 열기]({source_path.as_uri()}#page={page['pdf_page']})"))
    print(page['raw_text'])
    print('자동 검토 신호:', page['quality_flags'])
if not case['evidence']:
    print('문서 밖 질문입니다. 정답 페이지를 임의로 지정하지 않습니다.')


## 04. 검색과 답변 생성

모범답안은 생성 프롬프트에 넣지 않습니다. 모델에는 사용자 질문과 실제 검색한 청크만 전달합니다.
문서 밖 질문도 같은 검색/생성 경로로 보내므로 보류 동작을 실제로 관찰할 수 있습니다.
검색 결과가 있다는 이유로 정답이 있다고 판정하지 않습니다.

실패한 실행을 빈 답변으로 채점하지 않습니다. 네트워크/키/모델 문제를 해결한 뒤 재실행하세요.
재실행할 때 기존 완료 캐시와 다른 실험을 섞지 않습니다.


In [ ]:
if RUN_LIVE and not INDEX_READY:
    raise RuntimeError('현재 FAISS 인덱스가 없습니다. pipeline.pdf_pipeline build를 먼저 실행하세요.')
if RUN_LIVE and not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY가 없습니다. 프로젝트 .env를 설정하세요.')
RUN = ev.run_cases(CFG, QUESTIONS_ONLY, model=ANSWER_MODEL, top_k=TOP_K,
                   env_file=ENV_FILE if ENV_FILE.is_file() else None,
                   run_live=RUN_LIVE, use_cache=USE_CACHE) if INDEX_READY else None
if RUN is None:
    print('실행 결과가 없습니다. API 실행은 RUN_LIVE=True로 바꾼 뒤 이 셀부터 실행하세요.')
else:
    print('실행 ID:', RUN['run_id'])
    print('완료 사례:', len(RUN['items']))
    print('평가 방식:', RUN['grading'])

if RUN:
    import pandas as pd
    display(pd.DataFrame([{
        '질문': item['id'],
        '벡터 근거': len(item['retrieval']['results']),
        '그래프 근거': len(item['retrieval'].get('graph_results', [])),
        '최종 근거': len(ev.evidence_rows(item['retrieval'])),
        '웹 보완': item['retrieval'].get('safety_workflow', {}).get('web', {}).get('status', ''),
        '탐색 노드': item['retrieval'].get('graph_retrieval', {}).get('visited_node_count', 0),
        '상태': item['answer']['status'],
        '응답 시간(초)': item['latency_seconds'],
    } for item in RUN['items']]))


## 05. 모범답안, 생성 답변과 검색 근거 비교

좌측은 검토용 모범답안, 우측은 실제 답변입니다. 각 검색 결과를 펼쳐 원문을 확인하세요.
- candidate_hit_at_k / candidate_recall_at_k: 미확정 근거 후보에 대한 참고 수치입니다.
- citations_exist_in_context: 청크 ID 존재 여부입니다. 주장을 실제로 뒷받침하는지는 사람이 확인합니다.
- status_matches_draft: 모델의 상태와 예상 상태가 같은지만 확인합니다. 적절한 보류인지 의미 평가는 사람이 합니다.
- 문서 밖 질문의 검색 지표는 N/A입니다. 관련 없는 결과를 반환하는 것을 곧바로 검색 성공으로 보지 않습니다.

- combined_candidate_recall: 벡터·문서 연결·그래프를 합친 문맥의 근거 후보 포함률.
- graph_candidate_recall: 그래프만의 근거 후보 포함률. 두 값은 RAGAS 점수와 별도 진단입니다.
- S01/S12의 상태 비교는 공통 대응+추가 확인을 담은 answered 또는 needs_clarification을 허용합니다.


In [ ]:
COMPARISON_HTML = ev.comparison_html(CASES, RUN)
display(HTML(COMPARISON_HTML))
if RUN:
    OUTPUT_DIR = CFG['output_root'] / 'evaluation' / RUN['run_id']
    (OUTPUT_DIR / f'comparison-{REVIEW_VERSION}.html').write_text('<!doctype html><meta charset="utf-8">' + COMPARISON_HTML, encoding='utf-8')
    print('비교 화면 저장:', OUTPUT_DIR / f'comparison-{REVIEW_VERSION}.html')

if RUN:
    ev.write_json(OUTPUT_DIR / f'reference-snapshot-{REVIEW_VERSION}.json', {'cases': CASES, 'weights': ev.WEIGHTS, 'review_version': REVIEW_VERSION})


## 06. RAGAS 평가 — 실제 LLM 자동 채점

| 지표 | 의미 | 대상 |
|---|---|---|
| Faithfulness | 답변 주장이 검색 근거로 뒷받침되는 비율 | Q01~Q04, S01~S15 |
| Answer Relevancy | 답변이 질문에 대응하는 정도. 사실 정확성 지표는 아님 | Q01~Q04, S01~S15 |
| Context Precision (초안 기준) | 모범답안에 유용한 검색 구간이 상위에 있는 정도 | Q01~Q04, S01~S15 |
| Context Recall (초안 기준) | 모범답안의 주장을 검색 근거가 얼마나 포함하는지 | Q01~Q04, S01~S15 |
| 보류 응답 루브릭 1~5점 | 근거 없는 내용을 지어내지 않고 한계·추가 필요자료를 설명하는지 | Q05a |

RAGAS 0.4.3 collections API를 사용합니다. 서로 다른 지표를 임의로 합산해 정답률로 표현하지 않습니다.
평가는 질문·실제 답변·검색 PDF 구간·모범답안 초안을 OpenAI로 전송하며 비용이 발생합니다.
모범답안은 평가자에게만 전달하고 원래 답변 생성 입력에는 포함하지 않습니다.


In [ ]:
from pipeline.ragas_evaluation import evaluate_saved, report_path, publish
RUN_RAGAS = os.getenv('SAFEOPS_RUN_RAGAS', '0') == '1'  # True: 평가 API 호출. 저장된 완료 사례는 재사용.
RAGAS_REPORT = await evaluate_saved(RUN, CASES, live=RUN_RAGAS) if RUN else None
if RAGAS_REPORT:
    import pandas as pd
    table = pd.DataFrame([{'질문':r['id'], '평가 그룹':r['group'], '유형':r.get('category', ''), **r['scores']} for r in RAGAS_REPORT['rows']])
    display(table.fillna('해당 없음'))
    display(pd.DataFrame(RAGAS_REPORT.get('summary', {})).T.rename(columns={'mean':'평균', 'count':'평가 수'}))
    print('검색 정밀도·재현율은 미검토 모범답안 기준 잠정 점수. 사람 점수가 아닌 RAGAS LLM 채점입니다.')
    print('실행 기록:', report_path(RUN, CASES))
else:
    print('저장된 RAGAS 결과가 없습니다. RUN_RAGAS=True로 평가를 실행하세요.')

if RAGAS_REPORT and RAGAS_REPORT.get('by_category'):
    display(pd.DataFrame([{'유형': category, '지표': metric, **values} for category, metrics in RAGAS_REPORT['by_category'].items() for metric, values in metrics.items()]))

# 동일한 저장 답변의 한국어 역질문 진단: 기본 RAGAS 지표와 별도 비교한다.
from pipeline.ragas_evaluation import evaluate_relevance_language
LANGUAGE_REPORT = await evaluate_relevance_language(RUN, CASES, live=RUN_RAGAS) if RUN else None
if LANGUAGE_REPORT:
    standard = {r['id']: r['scores'].get('answer_relevancy') for r in (RAGAS_REPORT or {}).get('rows', [])}
    display(pd.DataFrame([{'질문': r['id'], '기본 관련성': standard.get(r['id']), '한국어 역질문 진단': r['answer_relevancy_ko']} for r in LANGUAGE_REPORT['rows']]))
    print('동일 답변의 평가 언어 차이를 보는 별도 진단입니다. 시스템 성능 개선 폭으로 해석하지 않습니다.')
    print('한국어 역질문 진단 평균:', LANGUAGE_REPORT['mean'])


### LangSmith RAGAS 실험

기존 후보 Hit/Recall 실험과 구분해 새 RAGAS 실험으로 등록합니다.
자동 업로드는 기본적으로 꺼져 있으며, 이미 저장한 실험 링크는 아래에 표시합니다.


In [ ]:
UPLOAD_RAGAS = False
if RAGAS_REPORT and RAGAS_REPORT.get('completed_at'):
    receipt_path = report_path(RUN, CASES).with_suffix('.receipt.json')
    if UPLOAD_RAGAS:
        receipt = publish(RUN, CASES, RAGAS_REPORT)
    elif receipt_path.exists():
        receipt = read_json(receipt_path)
    else:
        receipt = None
    if receipt:
        display(Markdown(f"[SafeOps RAGAS 평가 실험 열기]({receipt['experiment_url']})"))


## 07. 해석

- 높은 Faithfulness만으로 답변의 완전성이나 최신 법령 적합성을 보장하지 않습니다.
- Answer Relevancy는 임베딩 유사도를 사용하며 정답률이 아닙니다.
- Context Precision/Recall의 기준 답안은 현재 AI 초안입니다. 기준의 오류·누락은 점수에 영향을 줍니다.
- 보류 질문은 답을 제공하지 않는 것이 올바를 수 있어 일반 관련성 평균에 섞지 않습니다. 사용자 정의 루브릭은 RAGAS의 보편적인 표준 안전 점수가 아닙니다.
- 20문항은 개발용 검증 세트입니다. 점수와 함께 사례별 PDF 근거·실제 답변·평가 모델 및 버전을 확인하세요.
- 비교 화면의 후보 Hit/Recall과 인용 ID 존재 여부는 보조 진단이며 RAGAS 지표가 아닙니다.

- Q01 기준 답안에 43쪽 운반 지침을 추가했습니다. 이전 기준으로 계산한 점수는 이전 결과로 보존합니다.
- 기본 4문항과 새 상황 15문항의 결과를 유형별로 나누어 보세요.


## 08. 전처리·하이퍼파라미터 비교 실험

**목표: 최고 점수를 가정하지 않고, 같은 질문·모델로 변경 효과를 비교합니다.**

현재 분할기는 이미 제목/법률 조항을 먼저 구분하고 페이지 안에서 글자 수로 추가 분할합니다. 숫자는 **토큰 수가 아닌 문자 수**입니다.

1. 추출 품질 확인: 이미지/낮은 텍스트량 페이지와 원본·추출문 비교.
2. 무료 청크 비교: 크기 600/1000/1400, 중복 100/200 등. 원문 위치·후보 근거 포함·임베딩 토큰량 검사.
3. 유료 A/B: 기준 설정과 후보 하나만 비교. 임베딩 ada-002, 생성 gpt-4o, 평가 gpt-4o-mini 고정.
4. 우수 후보의 top_k 3/5/8 비교. 이후에만 임베딩 모델·검색 방식 변경을 별도 실험.
5. 최종 선택은 근거 충실도, 누락, 보류 응답, 문맥 길이·지연을 함께 확인. 자동 운영 반영하지 않음.

이 문항들은 **개발용**입니다. 이 질문으로 설정을 고른 뒤 같은 질문의 점수를 최종 일반화 성능으로 보고하면 안 됩니다. 별도의 보류 평가 질문을 준비하세요.


In [ ]:
from pipeline import tuning_experiments as tuning
from pipeline.pdf_pipeline import read_lines
import pandas as pd
PAGES = read_lines(resolve_pointer(CFG['output_root'], 'prepared.json') / 'pages.jsonl')
PAGE_AUDIT = pd.DataFrame(tuning.page_audit(PAGES))
display(PAGE_AUDIT.groupby('doc_id').agg(pages=('pdf_page','count'), image_pages=('image_flag','sum'), low_text_pages=('low_text','sum')))
display(PAGE_AUDIT[PAGE_AUDIT['low_text'] | PAGE_AUDIT['image_flag']])
print('표시된 페이지는 검토 후보입니다. 이미지 존재 ≠ OCR 필수, 텍스트량 많음 ≠ 추출 성공.')


### 08-1. 원본 이미지와 추출문 비교

아래 페이지를 바꿔 원본과 비교합니다. `layout` 추출도 완벽한 표 복원은 아닙니다.

| 문제 | 비교할 처리 | 확인 기준 |
|---|---|---|
| 스캔·이미지 글자 | 원래 추출 vs 한국어 OCR | 누락·오인식, 금지 표현, 수치·단위 |
| 표·다단 편집 | 기본 추출 vs layout/표 전용 파서 | 행·열 대응, 제목·각주 연결 |
| 흐린 작은 글자 | OCR 렌더링 200/300 DPI, 필요할 때 400 | 추가 비용·시간 대비 오인식 감소 |
| 혼합 페이지 | 텍스트는 보존하고 이미지 영역만 OCR | 같은 문장 중복 방지 |

OCR 결과는 원문 텍스트를 덮어쓰지 않고 별도 버전으로 저장해야 합니다. 엔진·언어·DPI·페이지·원본 해시를 기록하고, 도식/기호는 OCR만으로 해석됐다고 간주하지 않습니다.
**이 부록은 OCR을 실행하거나 OCR 엔진의 최적값을 검증하지 않습니다.** 먼저 아래 원본 비교로 적용 페이지를 선정합니다.


In [ ]:
from pypdf import PdfReader
from IPython.display import Image
import shutil, subprocess
REVIEW_DOC = 'lab_safety_manual'
REVIEW_PAGE = 12
source = next(d for d in CFG['documents'] if d['doc_id'] == REVIEW_DOC)
pdf_path = CFG['input_root'] / source['filename']
reader = PdfReader(pdf_path)
if not 1 <= REVIEW_PAGE <= len(reader.pages):
    raise ValueError('PDF 페이지 범위를 확인하세요.')
page = reader.pages[REVIEW_PAGE-1]
text_pair = {'기본 추출':page.extract_text() or '', 'layout 추출':page.extract_text(extraction_mode='layout') or ''}
preview_dir = CFG['output_root'] / 'tuning' / 'preview'
preview_dir.mkdir(parents=True, exist_ok=True)
if shutil.which('pdftoppm'):
    prefix = preview_dir / f'{REVIEW_DOC}-{REVIEW_PAGE}'
    subprocess.run(['pdftoppm','-f',str(REVIEW_PAGE),'-l',str(REVIEW_PAGE),'-singlefile','-scale-to','1200','-png',str(pdf_path),str(prefix)], check=True, capture_output=True)
    display(Image(filename=str(prefix)+'.png'))
else:
    print('원본 PDF를 직접 열어 비교하세요:', pdf_path)
for name,text in text_pair.items():
    display(Markdown(f'**{name} — {len(text)}자**'))
    print(text)


### 08-2. 비용 없는 청킹 비교

`candidate_evidence_containment`는 전체 청크 중 예상 근거 문구가 보존됐는지 검사합니다. **검색 Recall이나 RAGAS 점수가 아닙니다.**
`duplicated_character_ratio`는 원문 대비 청크 전체 문자량으로, 중복의 비용 영향을 보는 지표입니다. 낮다고 항상 좋은 것은 아닙니다.
`embedding_input_tokens`는 후보 전체를 임베딩할 때 입력되는 토큰의 추정치이며 달러 비용·생성 비용은 아닙니다.


In [ ]:
CHUNK_COMPARISON = pd.DataFrame(tuning.local_comparison(CFG, PAGES, CASES))
display(CHUNK_COMPARISON)
write_path = CFG['output_root'] / 'tuning' / 'chunk-comparison.csv'
write_path.parent.mkdir(parents=True, exist_ok=True)
CHUNK_COMPARISON.to_csv(write_path,index=False)
print('아직 최적 설정을 고르지 않습니다. 다음 검색·답변 평가가 필요합니다.')


### 08-3. 실제 검색·답변·RAGAS A/B 실험

기본값 False는 **저장된 결과만 조회**합니다. True이면 선택한 후보의 전체 PDF 청크를 OpenAI에 임베딩 전송하고 현재 질문 세트의 답변 생성·RAGAS 평가를 실행합니다. API 비용이 발생합니다.

- 기본 후보: 기존 1000/100 vs 1400/140. 누락 개선 여부를 검증하기 위한 가설이며 큰 청크가 좋다고 가정하지 않습니다.
- 기존 인덱스와 질문/답변 캐시는 재사용합니다. 새 후보 인덱스는 `pipeline_artifacts/tuning/` 아래 분리합니다.
- 실패 시 오류를 확인하고 재실행하세요. 완료된 RAGAS 사례는 재사용됩니다.
- 처음은 top_k=5로 고정합니다. 청킹 비교 후 선택한 후보 **하나**에 TOP_KS=[3,5,8], MAX_TRIALS=3으로 검색 수를 비교합니다.
- 새 결과의 LangSmith 업로드와 기존 활성 인덱스 교체는 자동 실행하지 않습니다.


In [ ]:
RUN_TUNING = False
SELECTED_VARIANTS = ['baseline_1000_100', 'large_1400_140']
TOP_KS = [5]
MAX_TRIALS = 2
TRIAL_RESULTS = await tuning.run_trials(CFG, PAGES, CASES, SELECTED_VARIANTS,
    top_ks=TOP_KS, live=RUN_TUNING, max_trials=MAX_TRIALS)
display(pd.DataFrame(TRIAL_RESULTS).fillna('미측정'))
print('미측정은 0점이 아닙니다. 기준 설정만 완료됐다면 후보를 최적이라고 결론낼 수 없습니다.')


### 08-4. 결과를 보고 선택하는 기준

1. **탈락 조건부터 확인**: 출처 유실, 수치/금지문구 누락, 문서 밖 질문에 근거 없는 확답. 평균 점수가 높아도 탈락시킵니다.
2. **RAGAS 비교**: 근거 충실도가 악화되지 않으면서 검색 재현율과 답변 관련성이 개선되는지 질문별로 봅니다. 기준 답안 초안의 오류도 함께 확인합니다.
3. **비용·응답시간 비교**: 비슷한 품질이면 임베딩·문맥 토큰과 지연이 작은 설정을 선택합니다. 캐시와 API 상태가 지연에 영향을 주므로 단일 측정으로 속도를 단정하지 않습니다.
4. **재현성 확인**: 상위 두 후보를 반복 평가해 점수 변동을 확인합니다. 현재 결과는 1회 평가입니다.
5. **보류 질문 검증**: 설정 선택에 쓰지 않은 질문으로 마지막 검증 후 적용합니다.

다음 실험은 한 축씩: 청킹 → top_k → 필요할 때 hybrid/MMR/reranker → 임베딩 모델. 임베딩 모델 변경 시 문서와 질문을 같은 모델로 임베딩하고 별도 인덱스를 만듭니다. 모델 간 원시 거리값을 같은 점수처럼 비교하지 않습니다.

**현재 상태: 추출/청크 진단 완료, 추가 유료 후보 실험은 미실행. 전역 최적 설정은 아직 확정하지 않음.**


## 09. 표와 GHS 그림 인식: 실제 두 단계 API 검증

**문제:** 텍스트 추출은 표의 글자를 읽지만 그림 종류를 놓칩니다. 범용 이미지 모델도 작은 그림을 누락할 수 있습니다.

**해결:** 기존 텍스트 + 전체 페이지·상단 확대 이미지로 표/NFPA/그림을 읽고, 빨간 테두리를 찾아 개별 그림을 확대해서 재분류합니다. 코드는 모델이 생성하지 않고 검증된 사전에서 연결합니다.

- 모델: `gpt-4o-2024-08-06`, temperature=0, image detail=high, 페이지 렌더링 180 DPI.
- 기준 자료: `books/data/SafeOps_GHS_인식참고_v1.pdf` (인식 참고 전용, RAG 색인 제외).
- 그림 출처는 매뉴얼 105쪽, 공식 코드 대조는 Safe Work Australia. 원문 번호 불일치를 보존합니다.
- 가성소다 PDF 54쪽/인쇄 53쪽, 암모니아 PDF 55쪽/인쇄 54쪽을 구분합니다.
- 비교 정답은 추출 후에만 읽습니다. 모델 프롬프트에 두 페이지 정답을 넣지 않습니다.
- 빨간 테두리 검출은 이 문서용 후보 검출기입니다. 흑백·훼손·다른 색상 기호에 일반화됐다고 가정하지 않습니다.
- 바운딩 박스 정확도는 아직 별도 정답 좌표로 측정하지 않았습니다. 화면에서 확인하세요.


In [ ]:
from pipeline.vision_extraction import recognize, refine_icons, compare
RUN_VISION = False  # True: 이미지와 추출문을 OpenAI에 전송, 판독 비용 발생
VISION_PAGES = [54, 55]  # 동의한 초기 검증 범위
FIRST_PASS = [recognize(page, live=RUN_VISION) for page in VISION_PAGES]
REFINED = [refine_icons(record, live=RUN_VISION) for record in FIRST_PASS]
display(Markdown('**1차: 전체 페이지 + 상단 확대 판독**'))
display(pd.DataFrame(compare(FIRST_PASS)))
display(Markdown('**2차: 빨간 테두리 검출 + 개별 그림 확대 분류**'))
display(pd.DataFrame(compare(REFINED)))
print('두 페이지 개발 검증 결과입니다. 9종 전체/다른 문서의 정확도 또는 운영 승인으로 해석하지 않습니다.')


In [ ]:
for record in REFINED:
    if record['status'] == 'not_run':
        continue
    display(Markdown(f"### PDF {record['pdf_page']}쪽 · 검토 대기"))
    display(pd.DataFrame([record['extracted']['fields']]))
    display(pd.DataFrame([record['extracted']['nfpa']]))
    display(pd.DataFrame(record['extracted']['icons']))
    print('실제 응답 ID:',record['response_id'])
    print('원본 결과:',record['result_path'])
    for region in record.get('regions',[]):
        print('검출 영역',region['region_id'])
        display(Image(filename=region['path'],width=180))


### 적용 전 확인

1. 두 페이지 모두 표 값과 그림 종류/개수를 사람이 확인합니다. 첫 판독의 누락 결과도 보존합니다.
2. 동일한 기호 사전 이미지 재인식은 독립 평가가 아닙니다. 실제 문서에서 나머지 기호·다른 해상도·흑백 사례를 모아 검증해야 합니다.
3. 작은 글자 오류가 남으면 먼저 180/300 DPI를 비교하고, 다음에 OCR 또는 표 전용 파서를 비교합니다. 여러 설정을 동시에 바꾸지 않습니다.
4. 승인된 구조화 결과만 별도 근거 청크로 만드는 다음 단계가 필요합니다. 현재 원본 텍스트·활성 인덱스는 변경하지 않았습니다.


## 10. 절차도와 비상연락망 구조 검토

원본과 사용자 설명을 대조한 전사 자료다. 절차 3개 경로와 연락망 8개 항목을 출처와 미확인 상태를 유지해 검색 DB에 반영했다. 좌표, 현행 번호와 담당자 직통번호는 미검증이다. 공휴일 기능은 사용자 요청으로 제거했다. 시간 선택은 한국 시간 월~금 09:00 이상 18:00 미만 주간, 그 외 야간이다. 이 셀들은 외부 API를 호출하지 않는다.

In [ ]:
from pipeline.safety_structure import load, validate, procedure_context, select_contacts
from datetime import datetime
import pandas as pd
from IPython.display import display
STRUCTURE_REPORT = validate()
display(STRUCTURE_REPORT)
display(pd.DataFrame([{'단계':n['node_id'],'원문':n['evidence']['raw_text'],'담당자':n['actor_raw'],'수신자':n['recipients_raw'],'기한':n['deadline_raw']} for n in load('procedure.json')['nodes']]))
display(pd.DataFrame([{'ID':c['contact_id'],'부서':c['organization_raw'],'업무':c['role_raw'],'원문 번호':c['phone_numbers_raw'],'원문 시간':c['availability_raw'],'번호 검증':c['current_validity']} for c in load('contacts.json')['contacts']]))

In [ ]:
# 2026-10-23 금요일 11:00 KST: 사용자 평일/주말 규칙으로 주간 선택.
CONTACT_DEMO = select_contacts('연구실 안전관리', 'lab_safety_manual_institution_unverified', datetime.fromisoformat('2026-10-23T11:00:00+09:00'))
display(CONTACT_DEMO)
display(procedure_context('right.submit'))

In [ ]:
import json
from pathlib import Path
STRUCTURE_QUESTIONS = json.loads((Path(ROOT) / 'evaluation/structure_questions.json').read_text())
display(pd.DataFrame(STRUCTURE_QUESTIONS['cases']))
# pass_criterion은 평가용이며, 실제 생성 답변이 아니다. 답변 생성 프롬프트에 넣지 않는다.

## 11. 통합 파이프라인 실제 검색 검증

379개 청크 인덱스의 실제 검색 결과와 가상 사고 입력으로 만든 근거 묶음이다. LLM 생성 답변이나 RAGAS 점수가 아니다. 평가 정답은 근거 묶음에 넣지 않는다. 이전 7문항 답변/평가 결과는 당시 인덱스에 대한 결과다.

In [ ]:
import json
from pathlib import Path
from IPython.display import display
import pandas as pd
INTEGRATED_CHECK = json.loads((Path(ROOT)/'pipeline_artifacts/structure-review/integrated-search-verification.json').read_text())
display({k:v for k,v in INTEGRATED_CHECK.items() if k!='checks'})
display(pd.DataFrame([{'질문':c['question'],'기대 페이지':c['expected_page'],'일치 순위':c['matching_ranks']} for c in INTEGRATED_CHECK['checks']]))

In [ ]:
from pipeline.incident_context import assemble
from pipeline.pdf_pipeline import config_at
from datetime import datetime
DEMO_FACTS=json.loads((Path(ROOT)/'evaluation/incident_demo_input.json').read_text())
INCIDENT_BUNDLE=assemble(config_at(Path(ROOT)/'pipeline/config.json'),None,DEMO_FACTS,datetime.fromisoformat('2026-10-23T11:00:00+09:00'))
display({k:v for k,v in INCIDENT_BUNDLE.items() if k not in ('evidence','contacts')})
display(pd.DataFrame([{'청크':r['chunk_id'],'페이지':r['pdf_page_start'],'종류':r.get('content_type','native_text'),'근거':r['text']} for r in INCIDENT_BUNDLE['evidence']]))
display(INCIDENT_BUNDLE['contacts'])

## 12. 문서 간 연결과 누락 방지 검증

물질 페이지, 공통 응급 지침, 사고 처리 절차, 연락망을 상황별로 연결했다. 아래는 실제 로컬 조합 결과이며 LLM 답변이나 실제 사고 기록이 아니다. 원문 적용 조건을 유지하며, 일반 지침과 물질별 지침의 충돌을 자동 해결했다고 보지 않는다. 화재 지침은 28쪽 제한조건과 31쪽 일반지침을 함께 제공한다. 전체 154페이지의 주제 후보 색인은 탐색용이며 모든 페이지의 의미 관계를 검증했다는 뜻이 아니다.

In [ ]:
from pipeline.pdf_pipeline import read_json,resolve_pointer,config_at
cfg=config_at(Path(ROOT)/'pipeline/config.json')
link_report=read_json(Path(ROOT)/'pipeline_artifacts/structure-review/document-link-verification.json')
display(link_report)
link_data=read_json(resolve_pointer(cfg['output_root'],'current.json')/'document_links.json')
display(pd.DataFrame(link_data['relations']))
display(pd.DataFrame(link_data['contact_observations']))

In [ ]:
# 가상 피부 오염 상황의 실제 근거 연결을 확인한다.
skin_bundle=read_json(Path(ROOT)/'pipeline_artifacts/structure-review/linked-scenario-skin.json')
display(pd.DataFrame([{'페이지':r['pdf_page_start'],'구간':r.get('section_id',r.get('content_type','native_text')),'출처문':r['text']} for r in skin_bundle['evidence']]))
print('확인 질문: 신체 오염 구간에 가스밸브 조작 문장이 섞이지 않았는가?')
print('확인 질문: 화재 일반지침과 소화전 사용 금지조건이 함께 제공되는가?')
print('확인 질문: 033-741-0114는 원문 표기이고, 2796과 760-2796의 동일성은 미확인으로 남았는가?')
print('확인 질문: 노출 경로 미상인 상황을 피부 노출로 추정하지 않았는가?')